<a href="https://colab.research.google.com/github/Locket51/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Locket51/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 119 (delta 31), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.84 MiB | 15.71 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/flyrank-ml-internship


## 1. My lane as an ML task (type)

**Task type: ranking (learned scoring).**

Underneath, this is a binary classification problem — I train a model to predict a proxy label. But the *task the user experiences* is ranking: the output is an ordered review queue, and what matters is that the pages near the top are the best candidates. That distinction matters because it changes the metric (top-K, not accuracy) and the way I validate. Lane 2 (Refresh / Content Opportunity Scoring) sits squarely in this category — the guide itself frames the deliverable as "a ranked action queue" with reason codes, not a set of predictions.

## 2. Target or proxy

**Proxy target:** `is_declining = (trend_direction == "down")`, stored as a 0/1 column.

**Where it comes from:** it is a *defined rule*, not a future observed outcome. `trend_direction` is a bucket computed from the current 90-day window in the starter slice — it compares recent movement to earlier movement inside the same snapshot. So it is a beginner proxy: cheap to build, honest to run, but not the ideal capstone target.

**What the ideal target would be:** a future outcome, e.g. features from the prior 90 days → did the page actually lose traffic over the next 30 days. That needs the warehouse daily table (fact_content_daily_performance) with a clean feature window and a separate target window, plus a leakage audit. For Week 2 I use the proxy so the loop runs end to end; the capstone upgrades the label to a future-window outcome.

In [2]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# sketch the target column: this is the proxy label
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining"].value_counts())
print("\npositive rate:", round(df["is_declining"].mean(), 3))
print("\nsample rows:")
print(df[["content_id", "trend_direction", "is_declining"]].head())

is_declining
1    16262
0    13738
Name: count, dtype: int64

positive rate: 0.542

sample rows:
             content_id trend_direction  is_declining
0  content_304f48230142            down             1
1  content_a1fb4e703a9e            down             1
2  content_9aa793d4d895            down             1
3  content_331d6c4de07b          stable             0
4  content_d99b7a2d90ca            down             1


## 3. Success metric

**Primary metric: Precision@50.**

Precision@50 answers the only question the reviewer actually cares about: *of the top 50 pages we tell them to open first, how many really are declining?* If the reviewer can act on 50 pages a week, that is the number that decides whether the tool is worth using.

**Why not accuracy:** the classes are roughly 54 / 46 (from Week 1's numbers), so a model that predicts "declining" for everything already gets ~54% accuracy while being useless as a ranking. Accuracy hides failure; Precision@50 exposes it.

**Supporting metrics I will also report:**
- **Average precision** — quality of the whole ranking, not just the top slice.
- **ROC AUC** — how well the model separates the two classes overall.
- **Precision@20** — if the reviewer only has time for 20 pages.

**What "good" looks like:** the starter baseline got Precision@50 = 0.240 and the random forest got 0.740. My bar for Week 2 is simply to *reproduce the loop and beat the baseline on the same slice*, not to hit a specific number yet. The exact threshold comes later, once I have the ranking running on the full warehouse.

In [3]:
# show why accuracy is a bad metric for this label
pos_rate = df["is_declining"].mean()
print(f"positive rate = {pos_rate:.3f}")
print(f"accuracy of a trivial 'always declining' classifier = {pos_rate:.3f}")
print("→ accuracy is ~0.54 for a useless model, so I use Precision@K instead")

positive rate = 0.542
accuracy of a trivial 'always declining' classifier = 0.542
→ accuracy is ~0.54 for a useless model, so I use Precision@K instead


## 4. The unit of analysis, as a real dataframe

**One row = one content page.**

The key is `content_id`. Each row is a single piece of content on a single client's site, summarised over the last 90 days. That is the grain the reviewer works at — they open a page, look at it, decide what to do with it. So the grain of my analysis has to match the grain of the decision.

**Column roles (quick map):**
- **Keys:** `content_id`, `client_id` — for joining and group-based validation, never as features.
- **Features (observable signals known before the decision):** `impressions_90d`, `clicks_90d`, `sessions_90d`, `ctr`, `avg_position`, `content_age_days`, `days_since_last_update`, `word_count`, `engagement_rate`, `scroll_rate`, plus the `*_tier` categoricals.
- **Label (proxy):** `is_declining`.
- **Excluded / leakage risk:** anything computed *from* `trend_direction` or `trend_pct` must not become a feature, because that's the label. `trend_pct` stays as a descriptive column only.

In [4]:
# one row = one content page
print("rows:", len(df))
print("unique content_id:", df["content_id"].nunique())
print("unique client_id:", df["client_id"].nunique())
print("→ grain: 1 row per content_id" if len(df) == df["content_id"].nunique() else "→ not 1:1, deduplicate first")

# show the slice that matches the lane (drop rows with no demand)
lane_slice = df[df["impressions_90d"] > 0].copy()
print("\nlane slice shape:", lane_slice.shape)

# the dataframe as the reviewer would see it
cols_to_show = [
    "content_id", "client_id", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "content_age_days", "days_since_last_update",
    "trend_direction", "is_declining",
]
print("\nhead of the lane slice:")
print(lane_slice[cols_to_show].head())

rows: 30000
unique content_id: 30000
unique client_id: 32
→ grain: 1 row per content_id

lane slice shape: (30000, 45)

head of the lane slice:
             content_id          client_id  impressions_90d  clicks_90d   ctr  \
0  content_304f48230142  client_f369cb89fc             3803          29  0.76   
1  content_a1fb4e703a9e  client_4e07408562            15320           7  0.05   
2  content_9aa793d4d895  client_7f2253d7e2            12581          11  0.09   
3  content_331d6c4de07b  client_19581e27de            11751          58  0.49   
4  content_d99b7a2d90ca  client_3fdba35f04            19140          24  0.13   

   avg_position  content_age_days  days_since_last_update trend_direction  \
0          10.6               187                      20            down   
1          20.3               445                      25            down   
2          36.5               141                      20            down   
3           6.2               463                      22    

## 5. Why ML beats a fixed rule here

The starter baseline already uses a fixed rule — a hand-tuned weighted sum:

    0.40 * visibility + 0.30 * freshness_risk + 0.25 * position_opportunity + 0.05 * depth_gap

That rule is useful and I will keep it as the baseline to beat. But it can only express what its author already knew to weight. It cannot capture interactions like:

- a page that is young *and* high-position *and* low-CTR, where the problem is snippet quality, not staleness;
- a page that is old *and* declining *and* still above 10k impressions, where refresh pay-off is high because volume is high;
- a page with strong CTR but collapsing scroll rate, where the issue is on-page, not the meta description.

A learned model can combine these signals into a non-linear score, learn how much each one actually matters on real outcomes, and produce a graded ranking with reason codes — while still leaving the reviewer in charge. The starter results already show the headroom: the rule scored Precision@50 = 0.240, a random forest reached 0.740 on the same slice. That gap is the case for ML over a fixed rule.

Two honesty notes:
- The gap may shrink on the full warehouse. The starter is a 30k slice, not a benchmark.
- The model is *decision-support*, not a guarantee — it ranks candidates, it does not prove that editing a page will cause a recovery.

In [5]:
# how varied is the lane slice across the signals the rule uses?
check_cols = ["impressions_90d", "ctr", "avg_position",
              "content_age_days", "days_since_last_update"]
print(lane_slice[check_cols].describe().round(2))
print("\ncorrelation of a few signals with the proxy label:")
print(lane_slice[["impressions_90d", "ctr", "avg_position",
                  "content_age_days", "is_declining"]]
      .corr(numeric_only=True)["is_declining"].round(3))

       impressions_90d       ctr  avg_position  content_age_days  \
count         30000.00  30000.00      30000.00          30000.00   
mean           5200.37      0.51         16.34            256.17   
std           16838.02      3.28         15.22            132.71   
min               1.00      0.00          0.00             90.00   
25%              81.00      0.00          6.20            132.00   
50%             731.00      0.07         10.80            236.00   
75%            3615.25      0.29         22.30            333.00   
max          517715.00    100.00        245.00            564.00   

       days_since_last_update  
count                30000.00  
mean                    46.10  
std                     42.08  
min                      1.00  
25%                     20.00  
50%                     20.00  
75%                    104.00  
max                    373.00  

correlation of a few signals with the proxy label:
impressions_90d    -0.018
ctr                -0

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card